# TP2 Violation Detection — SPX Equity Replication + DS-CAT Transformer

**Paper:** Glasserman & Pirjol (2026) — *When Are Option Prices TP₂?*  
**Dataset:** SPX (S&P 500) options via Databento OPRA, ~24 months EOD data  

---
**Running in VS Code + Colab extension:**
1. Click the Colab icon in VS Code (kernel picker → "Connect to Colab")
2. In Colab: Runtime → Change runtime type → **T4 GPU**
3. Run **Cell 1** (first code cell) — mounts Drive and sets up paths
4. Then run all remaining cells top to bottom

**Files to upload to Google Drive first (`MyDrive/tp2_project/` folder):**
```
MyDrive/tp2_project/
  tp2_project_code.zip     ← 84 KB  (the code)
  tp2_equity.db            ← 448 MB (pipeline data: snapshots, violations, reversions)
  equity_dataset.npz       ← 154 MB (pre-built ML features — skips 3-min build step)
  equity_metadata.json     ←  49 MB (sample metadata)
```

In [ ]:
# ── CELL 1: Mount Drive & Setup (run this first) ──────────────
import os, sys

# Reliable Colab detection — works on both web and VS Code extension
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

DRIVE_FILES = None  # set below if in Colab

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE = '/content/drive/MyDrive'
    DRIVE_FILES = f'{DRIVE}/tp2_project'   # subfolder where files were uploaded
    PROJECT_PATH = '/content/tp2_project'

    # Unzip code (only if not already done)
    if not os.path.isdir(PROJECT_PATH):
        print('Unzipping project code...')
        os.makedirs(PROJECT_PATH, exist_ok=True)
        ret = os.system(f'unzip -q "{DRIVE_FILES}/tp2_project_code.zip" -d "{PROJECT_PATH}"')
        if ret != 0:
            raise RuntimeError(f'unzip failed — is tp2_project_code.zip in {DRIVE_FILES}?')
        print('Done.')
    else:
        print('Project already unzipped.')

    # Create data directories
    os.makedirs(f'{PROJECT_PATH}/data/db', exist_ok=True)
    os.makedirs(f'{PROJECT_PATH}/data/processed', exist_ok=True)

    # Symlink large files from Drive → project (avoids copying 600 MB)
    DB_PATH   = f'{PROJECT_PATH}/data/db/tp2_equity.db'
    NPZ_PATH  = f'{PROJECT_PATH}/data/processed/equity_dataset.npz'
    META_PATH = f'{PROJECT_PATH}/data/processed/equity_metadata.json'

    for src, dst in [
        (f'{DRIVE_FILES}/tp2_equity.db',        DB_PATH),
        (f'{DRIVE_FILES}/equity_dataset.npz',   NPZ_PATH),
        (f'{DRIVE_FILES}/equity_metadata.json', META_PATH),
    ]:
        if os.path.exists(dst) or os.path.islink(dst):
            print(f'Already linked: {os.path.basename(dst)}')
        else:
            if not os.path.exists(src):
                raise FileNotFoundError(f'Not found in Drive: {src}')
            os.symlink(src, dst)
            print(f'Linked: {os.path.basename(dst)}')

else:
    # Local Mac
    PROJECT_PATH = '/Users/omkarter/Documents/Machine Learning in Quant Finance/Project/tp2_project'
    DB_PATH      = PROJECT_PATH + '/data/db/tp2_equity.db'
    NPZ_PATH     = PROJECT_PATH + '/data/processed/equity_dataset.npz'
    META_PATH    = PROJECT_PATH + '/data/processed/equity_metadata.json'

os.environ['TP2_ASSET_CLASS'] = 'equity'
sys.path.insert(0, PROJECT_PATH)
os.chdir(PROJECT_PATH)

print(f'\nPROJECT_PATH : {PROJECT_PATH}')
print(f'DB exists    : {os.path.exists(DB_PATH)}')
print(f'NPZ exists   : {os.path.exists(NPZ_PATH)}')

In [ ]:
# Install dependencies — only needed first time on Colab
if IN_COLAB:
    import subprocess
    subprocess.run(['pip', 'install', 'yfinance', 'seaborn', '--quiet'], check=True)

import torch, numpy, yfinance
print(f'torch {torch.__version__} | numpy {numpy.__version__} | yfinance {yfinance.__version__}')

if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
import sqlite3, logging
logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

import config as cfg
print(f'Asset class    : {cfg.ASSET_CLASS}')
print(f'Reversion fast : ≤{cfg.REVERSION_FAST_H}h ({int(cfg.REVERSION_FAST_H/24)} trading days)')
print(f'Reversion never: >{cfg.REVERSION_NEVER_H}h ({int(cfg.REVERSION_NEVER_H/24)} trading days)')

## 1. DB State Check

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

n_snap       = conn.execute('SELECT COUNT(*) FROM snapshots').fetchone()[0]
n_days       = conn.execute('SELECT COUNT(DISTINCT snapshot_hour) FROM snapshots').fetchone()[0]
n_clean      = conn.execute('SELECT COUNT(*) FROM snapshots WHERE is_clean=1').fetchone()[0]
n_violations = conn.execute('SELECT COUNT(*) FROM violations').fetchone()[0]
n_reversions = conn.execute('SELECT COUNT(*) FROM reversions').fetchone()[0]

row = conn.execute('SELECT MIN(snapshot_hour), MAX(snapshot_hour) FROM snapshots').fetchone()

print(f'=== EQUITY DB STATE ===')
print(f'Date range   : {row[0][:10]} → {row[1][:10]}')
print(f'Trading days : {n_days:,}')
print(f'Snapshots    : {n_snap:,}')
print(f'Clean rows   : {n_clean:,}')
print(f'Violations   : {n_violations:,}')
print(f'Reversions   : {n_reversions:,}')

# Expected values (pre-run locally)
assert n_violations > 300000, f'Expected 321K violations, got {n_violations} — DB may be incomplete'
assert n_reversions > 300000, f'Expected 320K reversions, got {n_reversions} — run labeler first'
print('\n✓ DB looks complete')

conn.close()

## 2. Paper Replication Check

Verify our SPX results match Glasserman & Pirjol (2026) Section 9:  
- Violations are **infrequent**  
- Most violations **reverse quickly** (within a few trading days)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

conn = sqlite3.connect(DB_PATH)
conn.row_factory = sqlite3.Row

n_viols = conn.execute('SELECT COUNT(*) FROM violations').fetchone()[0]
n_days  = conn.execute('SELECT COUNT(DISTINCT snapshot_hour) FROM snapshots WHERE is_clean=1').fetchone()[0]
print(f'=== PAPER REPLICATION: TP₂ EMPIRICAL RESULTS ===')
print(f'Trading days  : {n_days}')
print(f'Violations    : {n_viols:,}  ({n_viols/n_days:.0f}/day avg)')

speed_rows = conn.execute("""
    SELECT revert_speed, COUNT(*) as n FROM reversions
    WHERE revert_speed IN ('superfast','fast','medium','slow','never')
    GROUP BY revert_speed
    ORDER BY CASE revert_speed
        WHEN 'superfast' THEN 1 WHEN 'fast' THEN 2 WHEN 'medium' THEN 3
        WHEN 'slow' THEN 4 WHEN 'never' THEN 5 END
""").fetchall()

print(f'\n=== REVERSION SPEED ===')
total = sum(r['n'] for r in speed_rows)
for r in speed_rows:
    tag = ' ← label=1' if r['revert_speed'] in ('superfast','fast') else ''
    print(f'  {r["revert_speed"]:12s}: {r["n"]:6,}  ({r["n"]/max(total,1)*100:5.1f}%){tag}')

n_fast = sum(r['n'] for r in speed_rows if r['revert_speed'] in ('superfast','fast'))
print(f'\nFast rate : {n_fast/max(total,1)*100:.1f}%  | Paper: violations typically reverse quickly ✓')

# Plot violations per month
viol_by_month = conn.execute("""
    SELECT substr(snapshot_hour,1,7) as ym, COUNT(*) as n
    FROM violations GROUP BY ym ORDER BY ym
""").fetchall()

if viol_by_month:
    months = [r['ym'] for r in viol_by_month]
    counts = [r['n'] for r in viol_by_month]
    fig, ax = plt.subplots(figsize=(13, 3))
    ax.bar(range(len(months)), counts, color='steelblue', alpha=0.7)
    ax.set_xticks(range(0, len(months), 2))
    ax.set_xticklabels([months[i] for i in range(0, len(months), 2)], rotation=45, ha='right')
    ax.set_title('TP₂ Violations per Month — SPX (Glasserman & Pirjol 2026 Replication)')
    ax.set_ylabel('Violations'); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    os.makedirs('results', exist_ok=True)
    plt.savefig('results/spx_violations_per_month.png', dpi=150)
    plt.show()

conn.close()

## 3. Load Pre-Built Dataset

**v2 dataset** — replaces 5 dead features (always-zero in equity ohlcv) with:
- `maturity_gap` (fast=229d vs medium=298d — most discriminative)
- `moneyness_K1`, `moneyness_K2` (fast violations are more ATM)
- `strike_gap_pct` (fast=0.039 vs medium=0.078)
- `log_violation_pct` (normalized magnitude)

Upload `equity_dataset.npz` after rebuilding locally with `python -m ml.build_equity_dataset`.

In [ ]:
import json

NPZ_PATH  = PROJECT_PATH + '/data/processed/equity_dataset.npz'
META_PATH = PROJECT_PATH + '/data/processed/equity_metadata.json'

if os.path.exists(NPZ_PATH):
    print('Loading pre-built dataset from .npz...')
    npz = np.load(NPZ_PATH)
    with open(META_PATH) as f:
        metadata = json.load(f)
    dataset_dict = {
        'temporal':     npz['temporal'],
        'surface':      npz['surface'],
        'surface_mask': npz['surface_mask'],
        'labels':       npz['labels'],
        'metadata':     metadata,
    }
    print('Loaded from .npz (fast path)')
else:
    # Fallback: build from DB (~3 min)
    print('.npz not found — building from DB (~3 min)...')
    from ml.build_equity_dataset import build_equity_dataset
    dataset_dict = build_equity_dataset(db_path=DB_PATH)

D_TEMPORAL, D_SURFACE = 19, 8
labels = dataset_dict['labels']
print(f'Samples        : {len(labels):,}')
print(f'Label 1 (fast) : {(labels==1).sum():,}  ({(labels==1).mean()*100:.2f}%)')
print(f'Label 0 (slow) : {(labels==0).sum():,}')
print(f'Temporal shape : {dataset_dict["temporal"].shape}')
print(f'Surface shape  : {dataset_dict["surface"].shape}')

## 4. Chronological Split

70% train | 15% val | 15% test — always by time, never random shuffle.

In [ ]:
from ml.dataset import chronological_split, make_dataloaders

train_ds, val_ds, test_ds, norm_stats = chronological_split(dataset_dict)

# pos_neg_ratio=0.2: 20% positives per batch (less aggressive than 0.3)
# improves precision — previous run had too many FPs (5635 FP vs 347 TP)
loaders = make_dataloaders(train_ds, val_ds, test_ds, batch_size=128, balanced=True, pos_neg_ratio=0.2)

print(f'Train : {len(train_ds):,}  pos={train_ds.labels.sum():,} ({train_ds.labels.mean()*100:.1f}%)')
print(f'Val   : {len(val_ds):,}  pos={val_ds.labels.sum():,} ({val_ds.labels.mean()*100:.1f}%)')
print(f'Test  : {len(test_ds):,}  pos={test_ds.labels.sum():,} ({test_ds.labels.mean()*100:.1f}%)')

## 5. Build DS-CAT Model

```
Temporal (60×19) → Linear → PosEnc → TransformerEncoder ─┐
                                                           ├→ CrossAttn → Pool → MLP → P(fast)
Surface  (300×8) → Linear          → TransformerEncoder ─┘
```

`d_model=64, n_layers=2` (~150K params) — smaller model reduces overfitting with only ~3K positives in training.

In [ ]:
import torch
from ml.transformer import build_model

# Back to d_model=64, 2 layers (~150K params) — 1.3M was overfitting with only ~3K positives
model, device = build_model(
    d_temporal=D_TEMPORAL,
    d_surface=D_SURFACE,
    d_model=64,
    n_heads=4,
    n_layers=2,
    dropout=0.3,   # increased from 0.2 — regularise harder
)
print(f'Parameters : {model.count_parameters():,}')
print(f'Device     : {device}')

if device == 'cuda':
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM       : {vram:.1f} GB')

## 6. Train DS-CAT

- **Focal Loss** γ=1.5, alpha auto-computed (capped 0.6–0.9)
- **Balanced sampler** — 20% positives per batch
- **CosineAnnealingLR** — starts at 3e-4, decays to 3e-6 over 40 epochs
- **Early stopping DISABLED** — val period (Oct–Dec 2025) is a harder regime than test; val F1 stays flat ~0.12 regardless of model quality, so it's not a reliable stopping signal
- Checkpoint saved after every epoch; final checkpoint used for test evaluation

In [ ]:
os.makedirs('results', exist_ok=True)
os.makedirs('checkpoints/equity', exist_ok=True)

from ml.train import train

results = train(
    model=model,
    loaders=loaders,
    device=device,
    epochs=40,
    lr=3e-4,
    gamma=1.5,
    alpha=None,
    patience=None,   # DISABLED — val period is a harder regime, val F1 is unreliable
    save_dir='checkpoints/equity',
)

print(f'\nBest val F1 : {results["best_val_f1"]:.4f}')
if results['test_metrics']:
    m = results['test_metrics']
    print(f'Test F1={m["f1"]:.4f}  P={m["precision"]:.4f}  R={m["recall"]:.4f}  Acc={m["accuracy"]:.4f}')
    print(f'Threshold  : {m.get("threshold", 0.5):.2f}')
    print(f'TP={m["tp"]}  FP={m["fp"]}  FN={m["fn"]}  TN={m["tn"]}')

## 7. Evaluate & Plot

In [ ]:
import matplotlib.pyplot as plt

history = results['history']
ep = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(ep, history['train_loss'], label='Train')
if history['val_loss']:
    axes[0].plot(ep, history['val_loss'], label='Val')
axes[0].set_title('Focal Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

if history['val_f1']:
    axes[1].plot(ep, history['val_f1'], color='green')
    best_ep = history['val_f1'].index(max(history['val_f1'])) + 1
    axes[1].axvline(best_ep, color='red', linestyle='--', alpha=0.7, label=f'Best ep {best_ep}')
    axes[1].set_ylim(0, 1)
axes[1].set_title('Val F1'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history['lr'], color='orange')
axes[2].set_title('Learning Rate'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/training_curves_spx.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns

if results['test_metrics']:
    m = results['test_metrics']
    cm = [[m['tn'], m['fp']], [m['fn'], m['tp']]]
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred Slow','Pred Fast'],
                yticklabels=['True Slow','True Fast'])
    ax.set_title(f'SPX TP₂ — DS-CAT\nF1={m["f1"]:.4f}  P={m["precision"]:.4f}  R={m["recall"]:.4f}')
    plt.tight_layout()
    plt.savefig('results/confusion_matrix_spx.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Copy checkpoint back to Drive so it persists after Colab session ends
if IN_COLAB and DRIVE_FILES:
    import shutil
    ckpt_src = 'checkpoints/equity/best_model.pt'
    ckpt_dst = f'{DRIVE_FILES}/ds_cat_spx_best_model.pt'
    if os.path.exists(ckpt_src):
        shutil.copy(ckpt_src, ckpt_dst)
        print(f'Checkpoint saved to Drive: {ckpt_dst}')

if results['test_metrics']:
    m = results['test_metrics']
    n_params = model.count_parameters()
    print('=' * 50)
    print('   DS-CAT SPX RESULTS — Glasserman & Pirjol 2026')
    print('=' * 50)
    print(f'  Data   : SPX options, 487 trading days (~24 months)')
    print(f'  Model  : DS-CAT ({n_params:,} parameters)')
    print(f'  Task   : Predict fast TP₂ reversion (≤2 trading days)')
    print(f'  ------')
    print(f'  F1     : {m["f1"]:.4f}')
    print(f'  P / R  : {m["precision"]:.4f} / {m["recall"]:.4f}')
    print(f'  Acc    : {m["accuracy"]:.4f}')
    print(f'  Thresh : {m.get("threshold", 0.5):.2f}')
    print(f'  TP={m["tp"]}  FP={m["fp"]}  FN={m["fn"]}  TN={m["tn"]}')
    print('=' * 50)
    print('\nNext: load this checkpoint for crypto (BTC/ETH/SOL) transfer learning')